# Togather — Analysis

**Goal:** Identify growth opportunities and commercial recommendations for 2025.  
**Builds on:** `01_eda.ipynb` — run that first to understand data quality issues.

## 0. Setup

In [ ]:
import warnings
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.dpi'] = 120

FIGURES = '../outputs/figures/'
RAW     = '../data/raw/'

In [ ]:
def load_requests(path=RAW + 'food_requests.xlsx'):
    df = pd.read_excel(path, header=1)
    df.columns = ['event_request_id', 'created', 'region', 'priority_tag', 'request_budget']
    df['created'] = pd.to_datetime(df['created'], format='mixed')
    df['priority_tag'] = df['priority_tag'].str.strip()
    return df

def load_quotes(path=RAW + 'food_quotes.xlsx'):
    df = pd.read_excel(path)
    df.columns = [
        'quote_id', 'supplier_id', 'event_request_id',
        'quote_created', 'booked', 'supplier_region',
        'quote_price', 'supplier_primary_tag'
    ]
    df['quote_created'] = pd.to_datetime(df['quote_created'], format='mixed')
    df['booked'] = df['booked'].astype(int)
    df['supplier_primary_tag'] = df['supplier_primary_tag'].str.strip()
    return df

requests = load_requests()
quotes   = load_quotes()
print('Requests:', requests.shape)
print('Quotes:  ', quotes.shape)

---
## 1. Macro Category Mapping

In [ ]:
TAG_GROUPS = {
    # Street Food
    'Pizza':          'Street Food',
    'Burgers':        'Street Food',
    'Hot Dogs':       'Street Food',
    'Fries':          'Street Food',
    'Wraps':          'Street Food',
    'Sandwiches':     'Street Food',
    'Fish and Chips': 'Street Food',
    'Fried Chicken':  'Street Food',
    'Mac & Cheese':   'Street Food',
    # BBQ & Meats
    'BBQ':          'BBQ & Meats',
    'Hog Roast':    'BBQ & Meats',
    'Pulled Meats': 'BBQ & Meats',
    'Pies':         'BBQ & Meats',
    # Asian & Indian
    'Asian':      'Asian & Indian',
    'Chinese':    'Asian & Indian',
    'Indian':     'Asian & Indian',
    'Japanese':   'Asian & Indian',
    'Thai':       'Asian & Indian',
    'Vietnamese': 'Asian & Indian',
    'Korean':     'Asian & Indian',
    'Curries':    'Asian & Indian',
    # World & Fusion
    'Mexican':        'World & Fusion',
    'Caribbean':      'World & Fusion',
    'South American': 'World & Fusion',
    'African':        'World & Fusion',
    'Middle Eastern': 'World & Fusion',
    'Fusion':         'World & Fusion',
    'Seafood':        'World & Fusion',
    # European
    'Italian':        'European',
    'French':         'European',
    'Greek':          'European',
    'Spanish':        'European',
    'Mediterranean':  'European',
    'Pasta':          'European',
    'Modern British': 'European',
    'Fine dining':    'European',
    # Sweet & Desserts
    'Dessert':             'Sweet & Desserts',
    'Ice Cream':           'Sweet & Desserts',
    'Waffles':             'Sweet & Desserts',
    'Doughnuts & Churros': 'Sweet & Desserts',
    'Cr\u00eapes':         'Sweet & Desserts',
    'Afternoon Tea':       'Sweet & Desserts',
    'Wedding Cake':        'Sweet & Desserts',
    # Sharing & Events
    'Canap\u00e9s':             'Sharing & Events',
    'Salads':                   'Sharing & Events',
    'Grazing boards or tables': 'Sharing & Events',
    'Gifting & Hampers':        'Sharing & Events',
    'Breakfast':                'Sharing & Events',
    'Seasonal':                 'Sharing & Events',
    'Festive':                  'Sharing & Events',
    # Dietary
    'Strictly Vegan': 'Dietary',
    'Vegan':          'Dietary',
    'Sustainable':    'Dietary',
    # Bar & Drinks
    'Indoor bar':         'Bar & Drinks',
    'Horsebox bar':       'Bar & Drinks',
    'Oh-wow outdoor bar': 'Bar & Drinks',
    'Outdoors':           'Bar & Drinks',
    'Rooftop':            'Bar & Drinks',
    'Beer & Cider':       'Bar & Drinks',
    # Service
    'Waiters':    'Service',
    'Bartenders': 'Service',
    'Vehicle':    'Service',
    'Delivery':   'Service',
}

def apply_macro(tag):
    if pd.isna(tag):
        return None
    return TAG_GROUPS.get(str(tag).strip(), 'Other')

requests['macro_category'] = requests['priority_tag'].map(apply_macro)
quotes['macro_category']   = quotes['supplier_primary_tag'].map(apply_macro)

# Coverage check
for name, df in [('requests', requests), ('quotes', quotes)]:
    mapped = df['macro_category'].notna() & (df['macro_category'] != 'Other')
    other  = (df['macro_category'] == 'Other').sum()
    null   = df['macro_category'].isna().sum()
    print(f'{name}: mapped={mapped.sum():,} ({mapped.mean():.1%}) | other={other:,} | null={null:,}')

print()
print('Unmapped (requests):', requests[requests['macro_category']=='Other']['priority_tag'].unique().tolist())
print('Unmapped (quotes):  ', quotes[quotes['macro_category']=='Other']['supplier_primary_tag'].unique().tolist())

---
## 2. Base Merge

One row per quote (requests with no quotes appear once with NaN quote fields).

In [ ]:
df = requests.merge(
    quotes.rename(columns={'macro_category': 'supplier_macro'}),
    on='event_request_id',
    how='left'
).rename(columns={'macro_category': 'customer_macro'})

# Convenience flags
df['has_quote']  = df['quote_id'].notna()
df['is_booked']  = df['booked'].fillna(0).astype(int)
df['month']      = df['created'].dt.to_period('M')
df['quarter']    = df['created'].dt.to_period('Q')

print('Merged shape:', df.shape)
print('Columns:', df.columns.tolist())
df.head(3)

---
## 3. Bank Holiday Flag

**Target variable for this section:** `is_booked` at **request level** — i.e., did a given event request result in at least one booking? This collapses the quote-level data to one row per request, which is the right unit for measuring whether a customer's need was fulfilled.

We attach the holiday flag to the **request creation date** — asking: *do requests created on or around bank holidays behave differently?*

In [ ]:
# Load bank holiday calendar
holidays_df = pd.read_csv(RAW + 'uk_bank_holidays_2024_2025.csv', parse_dates=['date'])
holidays_df['date'] = holidays_df['date'].dt.date

# Add is_bank_holiday to the merged df (based on request creation date)
df['request_date'] = df['created'].dt.date
df = df.merge(
    holidays_df[['date', 'is_bank_holiday_england', 'is_bank_holiday_scotland',
                 'holiday_name_england', 'is_weekend']].rename(columns={'date': 'request_date'}),
    on='request_date',
    how='left'
)

# Unified flag: bank holiday in England (covers most of the data)
df['is_bank_holiday'] = df['is_bank_holiday_england'].fillna(0).astype(int)

print('Bank holiday flag attached.')
print(f"  Rows on bank holidays: {df['is_bank_holiday'].sum():,}")
print(f"  Rows on normal days:   {(df['is_bank_holiday']==0).sum():,}")
print()
print('Unique bank holiday dates in dataset:')
print(df[df['is_bank_holiday']==1][['request_date','holiday_name_england']].drop_duplicates().to_string(index=False))

### 3.1 Effect of Bank Holidays on Bookings

We analyse three angles:
1. **Request volume** — do fewer/more requests come in on bank holidays?
2. **Booking conversion** — of requests created on bank holidays, do they convert differently?
3. **Quote response time** — do suppliers respond slower around bank holidays?

In [ ]:
# ── Collapse to request level (one row per event_request_id) ─────────────────
# Target: is_booked = 1 if the request resulted in ANY booking
req_level = (
    df.groupby('event_request_id').agg(
        created         = ('created', 'first'),
        region          = ('region', 'first'),
        priority_tag    = ('priority_tag', 'first'),
        customer_macro  = ('customer_macro', 'first'),
        request_budget  = ('request_budget', 'first'),
        request_date    = ('request_date', 'first'),
        is_bank_holiday = ('is_bank_holiday', 'first'),
        is_weekend      = ('is_weekend', 'first'),
        holiday_name    = ('holiday_name_england', 'first'),
        n_quotes        = ('quote_id', 'count'),
        is_booked       = ('is_booked', 'max'),   # 1 if any quote was booked
    ).reset_index()
)
req_level['has_quote'] = (req_level['n_quotes'] > 0).astype(int)

print('Request-level df shape:', req_level.shape)
print('Overall booking rate (request level):', req_level['is_booked'].mean().round(4))

# ── 1. Request volume on bank holidays vs normal days ────────────────────────
daily_vol = (
    req_level.groupby(['request_date', 'is_bank_holiday']).size()
    .reset_index(name='n_requests')
)
avg_vol = daily_vol.groupby('is_bank_holiday')['n_requests'].mean().rename({0: 'Normal day', 1: 'Bank holiday'})

print('\n--- Request volume (avg per day) ---')
print(avg_vol.round(1).to_string())
print(f'  Bank holidays receive {avg_vol[1]/avg_vol[0]:.1%} of a normal day volume')

# ── 2. Booking conversion rate on bank holidays vs normal days ────────────────
conv = req_level.groupby('is_bank_holiday')['is_booked'].mean().rename({0: 'Normal day', 1: 'Bank holiday'})
print('\n--- Booking conversion rate (request level) ---')
print(conv.apply(lambda x: f'{x:.2%}').to_string())
lift = (conv[1] - conv[0]) / conv[0]
print(f'  Lift on bank holidays: {lift:+.1%}')

# ── 3. Quote response time around bank holidays ───────────────────────────────
resp = df.merge(
    req_level[['event_request_id', 'is_bank_holiday']],
    on='event_request_id', how='left', suffixes=('', '_req')
).dropna(subset=['quote_created', 'created'])

resp['hours_to_quote'] = (resp['quote_created'] - resp['created']).dt.total_seconds() / 3600
resp = resp[(resp['hours_to_quote'] >= 0) & (resp['hours_to_quote'] <= 720)]  # cap at 30 days

resp_summary = resp.groupby('is_bank_holiday')['hours_to_quote'].median().rename({0: 'Normal day', 1: 'Bank holiday'})
print('\n--- Median hours to first quote ---')
print(resp_summary.round(1).to_string())

In [ ]:
# ── Visualise the three effects ───────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(14, 4))

# 1. Volume
avg_vol.plot(kind='bar', ax=axes[0], color=['steelblue', 'tomato'], legend=False)
axes[0].set_title('Avg daily request volume')
axes[0].set_ylabel('Requests per day')
axes[0].set_xticklabels(avg_vol.index, rotation=0)
for i, v in enumerate(avg_vol):
    axes[0].text(i, v + 0.5, f'{v:.0f}', ha='center', fontsize=9)

# 2. Conversion
conv_pct = conv * 100
conv_pct.plot(kind='bar', ax=axes[1], color=['steelblue', 'tomato'], legend=False)
axes[1].set_title('Booking conversion rate')
axes[1].set_ylabel('%')
axes[1].set_xticklabels(conv_pct.index, rotation=0)
for i, v in enumerate(conv_pct):
    axes[1].text(i, v + 0.05, f'{v:.2f}%', ha='center', fontsize=9)

# 3. Response time
resp_summary.plot(kind='bar', ax=axes[2], color=['steelblue', 'tomato'], legend=False)
axes[2].set_title('Median hours to first quote')
axes[2].set_ylabel('Hours')
axes[2].set_xticklabels(resp_summary.index, rotation=0)
for i, v in enumerate(resp_summary):
    axes[2].text(i, v + 0.5, f'{v:.0f}h', ha='center', fontsize=9)

plt.suptitle('Bank Holiday Effect on Platform Behaviour', fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig(FIGURES + '19_bank_holiday_effect.png', bbox_inches='tight')
plt.show()

# Per-holiday breakdown
print('--- Conversion rate per bank holiday ---')
per_hol = (
    req_level[req_level['is_bank_holiday'] == 1]
    .groupby('holiday_name')
    .agg(n_requests=('event_request_id','count'), booking_rate=('is_booked','mean'))
    .sort_values('booking_rate', ascending=False)
)
per_hol['booking_rate'] = per_hol['booking_rate'].apply(lambda x: f'{x:.2%}')
print(per_hol.to_string())